<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# Module 4.2: FIRRTL AST Traversal
**Prev: [Introduction to FIRRTL](4.1_firrtl_ast.ipynb)**<br>
**Next: [Common Pass Idioms](4.3_firrtl_common_idioms.ipynb)**

### From text to a tree
In the previous section we saw that Chisel emits FIRRTL as text. To analyze a design with a program, we want a data structure instead of a string: a tree whose nodes are circuits, modules, statements and expressions. Chisel 7 does not ship such a data structure (the FIRRTL compiler is now `firtool`, which is not a Scala program), so we will write a small one ourselves. This is a good exercise in Scala: case classes, pattern matching and the collection library are all that is needed.

Our AST covers the subset of FIRRTL that Chisel emits for ordinary designs: modules, ports, registers, wires, nodes, instances, connects, `when`/`else` and all expressions. Anything else (printf, memories, ...) is kept as raw text in an `OtherStmt`.

## Setup
If a cell fails or hangs, see [Troubleshooting](docs/troubleshooting.md).

In [ ]:
interp.load.module(os.pwd / "source" / "load-ivy.sc")

In [ ]:
import chisel3._
import chisel3.Input // avoid a name clash with almond.input.Input
import chisel3.util._

## The IR nodes
The case classes below are our IR. Their names follow the ones of the (former) Scala FIRRTL library. The following table shows the possible children of each IR node type:

```
+------------+-----------------------------+
|    Node    |          Children           |
+------------+-----------------------------+
| Circuit    | DefModule                   |
| DefModule  | Port, Statement             |
| Statement  | Statement, Expression       |
| Expression | Expression                  |
+------------+-----------------------------+
```

Types (`UInt<4>`, bundles, ...) are kept as plain strings, since we do not need to look inside them. Each statement also has an `info` field with its source locator (an empty string if there is none).

The `mapExpr` and `mapStmt` methods apply a function to the children of a node. We will explain them below.

In [ ]:
// ---- Expressions ----
sealed trait Expression {
  // Apply f to the child Expressions of this node (like firrtl's `map`).
  def mapExpr(f: Expression => Expression): Expression = this match {
    case DoPrim(op, args, consts) => DoPrim(op, args.map(f), consts)
    case leaf                     => leaf
  }
}
case class Reference(name: String)             extends Expression // x, io.in[0], m0.io.out
case class Literal(tpe: String, value: String) extends Expression // UInt<3>(0h5)
case class DoPrim(op: String, args: Seq[Expression], consts: Seq[Int])
    extends Expression // add(a, b), bits(a, 3, 0), mux(c, a, b)

// ---- Statements ----
sealed trait Statement {
  def info: String // source locator, e.g. "cmd3.sc 6:19" ("" if there is none)

  // Apply f to the Expressions directly contained in this statement.
  def mapExpr(f: Expression => Expression): Statement = this match {
    case s: DefRegister      => s.copy(clock = f(s.clock))
    case s: DefRegisterReset => s.copy(clock = f(s.clock), reset = f(s.reset), init = f(s.init))
    case s: DefNode          => s.copy(value = f(s.value))
    case s: Connect          => s.copy(loc = f(s.loc), expr = f(s.expr))
    case s: Conditionally    => s.copy(pred = f(s.pred))
    case other               => other
  }
  // Apply f to the child Statements of this statement.
  def mapStmt(f: Statement => Statement): Statement = this match {
    case s: Conditionally => s.copy(conseq = s.conseq.map(f), alt = s.alt.map(f))
    case s: OtherStmt     => s.copy(children = s.children.map(f))
    case other            => other
  }
}
sealed trait IsRegister extends Statement { def name: String; def tpe: String }
case class DefRegister(info: String, name: String, tpe: String, clock: Expression) extends IsRegister
case class DefRegisterReset(
    info: String,
    name: String,
    tpe: String,
    clock: Expression,
    reset: Expression,
    init: Expression
) extends IsRegister
case class DefWire(info: String, name: String, tpe: String)                                           extends Statement
case class DefNode(info: String, name: String, value: Expression)                                     extends Statement
case class DefInstance(info: String, name: String, module: String)                                    extends Statement
case class Connect(info: String, loc: Expression, expr: Expression)                                   extends Statement
case class Conditionally(info: String, pred: Expression, conseq: Seq[Statement], alt: Seq[Statement]) extends Statement
// Everything we do not model in detail (printf, assert, mem, ...). Keeps the raw text.
case class OtherStmt(info: String, text: String, children: Seq[Statement]) extends Statement

// ---- Modules and circuits ----
case class Port(info: String, direction: String, name: String, tpe: String)
case class DefModule(info: String, name: String, isPublic: Boolean, ports: Seq[Port], body: Seq[Statement]) {
  def mapStmt(f: Statement => Statement): DefModule = copy(body = body.map(f))
}
case class Circuit(main: String, modules: Seq[DefModule], otherDecls: Seq[String] = Nil) {
  def mapModule(f: DefModule => DefModule): Circuit = copy(modules = modules.map(f))
}

## A hand-made AST
Before parsing anything, we can build a tree by hand. This is the FIRRTL for `io.out := io.in + 1` in a hypothetical module, written as an AST:

In [ ]:
// Pretty-prints any of our case classes as an indented tree: one node per line.
def showTree(x: Any, depth: Int = 0, label: String = ""): String = {
  val pad                     = "  " * depth + (if (label.isEmpty) "" else label + ": ")
  def isNode(v: Any): Boolean = v match {
    case _: Expression | _: Statement | _: Port | _: DefModule | _: Circuit => true
    case _                                                                  => false
  }
  x match {
    case p: Product if isNode(p) =>
      val fields        = p.productElementNames.toList.zip(p.productIterator.toList)
      val (kids, atoms) = fields.partition {
        case (_, v: Seq[_]) => v.nonEmpty && v.forall(isNode)
        case (_, v)         => isNode(v)
      }
      val head = pad + p.productPrefix + atoms.filter(_._2 != "").map { case (n, v) => s" $n=$v" }.mkString
      (head +: kids.flatMap {
        case (n, v: Seq[_]) => v.map(showTree(_, depth + 1, n))
        case (n, v)         => Seq(showTree(v, depth + 1, n))
      }).mkString("\n")
    case other => pad + other.toString
  }
}

val addOne = DefModule(
  "",
  "AddOne",
  true,
  Seq(Port("", "input", "in", "UInt<4>"), Port("", "output", "out", "UInt<5>")),
  Seq(Connect("", Reference("out"), DoPrim("add", Seq(Reference("in"), Literal("UInt<1>", "0h1")), Nil)))
)

println(showTree(addOne))

`showTree` is our own pretty-printer. It prints one node per line, and the children are indented below their parent. Each case class also has the usual `toString`, but for large trees that is hard to read.

## Parsing FIRRTL
Now, let's fill the tree from text. The FIRRTL emitted by Chisel is line-oriented and indentation-based, so parsing goes in four steps:

1. Split the text into lines. Remember the indentation, and cut the source locator `@[...]` off the end of each line.
2. Use the indentation to arrange the lines into a tree: lines indented deeper than the previous one are its children.
3. Parse each line's text into an `Expression` or a `Statement`.
4. Put it together: skip the header and annotations, and parse each `module`.

Step 1 and 2:

In [ ]:
case class Line(indent: Int, text: String, info: String)
case class Node(line: Line, kids: Seq[Node])

val infoRegex = """^(.*?)\s*@\[(.*)\]\s*$""".r

def toLines(text: String): List[Line] =
  text.linesIterator
    .filter(_.trim.nonEmpty)
    .map { raw =>
      val indent = raw.length - raw.dropWhile(_ == ' ').length
      raw.trim match {
        case infoRegex(code, info) => Line(indent, code, info)
        case code                  => Line(indent, code, "")
      }
    }
    .toList

// Parses the sibling lines at `indent`; a more indented line starts the children of the line before it.
def toTree(lines: List[Line], indent: Int): (Vector[Node], List[Line]) = {
  var rest     = lines
  val siblings = Vector.newBuilder[Node]
  while (rest.nonEmpty && rest.head.indent == indent) {
    val line              = rest.head
    val (kids, remaining) =
      if (rest.tail.nonEmpty && rest.tail.head.indent > indent) toTree(rest.tail, rest.tail.head.indent)
      else (Vector.empty[Node], rest.tail)
    siblings += Node(line, kids)
    rest = remaining
  }
  (siblings.result(), rest)
}

val sample = """  module Foo : @[a.scala 1:1]
    input x : UInt<1>
    when x : @[a.scala 2:3]
      connect y, x @[a.scala 3:5]
    else :
      connect y, z"""
toTree(toLines(sample), 2)._1.foreach(println)

Step 3 starts with expressions. We split an expression into its operator and its arguments, taking care to split only at the commas that are not nested in parentheses, brackets, braces or angle brackets (think of `mux(c, add(a, b), d)` or the bundle type `{ a : UInt<1>, b : UInt<2> }`). Arguments that are plain integers are the constants of a primitive operation, like the `3` and `0` of `bits(x, 3, 0)`.

In [ ]:
// Split on commas that are not nested inside (), [], {} or <>.
def splitTop(s: String): List[String] = {
  var depth = 0
  val parts = scala.collection.mutable.ListBuffer[String]()
  val cur   = new StringBuilder
  for (c <- s) c match {
    case '(' | '[' | '{' | '<' => depth += 1; cur += c
    case ')' | ']' | '}' | '>' => depth -= 1; cur += c
    case ',' if depth == 0     => parts += cur.toString.trim; cur.clear()
    case _                     => cur += c
  }
  if (cur.nonEmpty) parts += cur.toString.trim
  parts.toList
}

val literalRegex = """^((?:UInt|SInt)(?:<\d+>)?)\((.*)\)$""".r
val primRegex    = """^([A-Za-z0-9_]+)\((.*)\)$""".r
val intRegex     = """^-?\d+$""".r

def parseExpr(text: String): Expression = text.trim match {
  case literalRegex(tpe, value) => Literal(tpe, value)
  case primRegex(op, args)      =>
    val (consts, exprs) = splitTop(args).partition(a => intRegex.matches(a))
    DoPrim(op, exprs.map(parseExpr), consts.map(_.toInt))
  case ref => Reference(ref)
}

println(showTree(parseExpr("mux(sel, add(a, UInt<3>(0h5)), bits(io.v[1], 3, 0))")))

Next, statements, ports and modules. Each kind of line is recognized by a regular expression that is matched with Scala's pattern matching. A `when` is the only statement with nested statements that we model, and it has to deal with its `else :`, which is a *sibling* line of the `when`, but belongs to it. That is why `toStatements` converts the siblings together.

In [ ]:
val portRegex     = """^(input|output) (\S+) : (.*)$""".r
val regRegex      = """^reg (\S+) : (.*)$""".r
val regResetRegex = """^regreset (\S+) : (.*)$""".r
val wireRegex     = """^wire (\S+) : (.*)$""".r
val nodeRegex     = """^node (\S+) = (.*)$""".r
val instRegex     = """^inst (\S+) of (\S+)$""".r
val connectRegex  = """^connect (.*)$""".r
val whenRegex     = """^when (.*) :$""".r
val moduleRegex   = """^(public )?module (\S+) :$""".r

// An `else :` node belongs to the `when` right before it, so we convert siblings together.
def toStatements(nodes: Seq[Node]): Seq[Statement] =
  nodes.foldLeft(Vector.empty[Statement]) {
    case (done, Node(Line(_, "else :", _), kids)) =>
      done.last match {
        case w: Conditionally => done.init :+ w.copy(alt = toStatements(kids))
        case _                => sys.error("`else` without `when`")
      }
    case (done, node) => done :+ toStatement(node)
  }

def toStatement(node: Node): Statement = {
  val Node(Line(_, text, info), kids) = node
  text match {
    case regResetRegex(name, rest) =>
      val Seq(tpe, clock, reset, init) = splitTop(rest)
      DefRegisterReset(info, name, tpe, parseExpr(clock), parseExpr(reset), parseExpr(init))
    case regRegex(name, rest) =>
      val Seq(tpe, clock) = splitTop(rest)
      DefRegister(info, name, tpe, parseExpr(clock))
    case wireRegex(name, tpe)    => DefWire(info, name, tpe)
    case nodeRegex(name, value)  => DefNode(info, name, parseExpr(value))
    case instRegex(name, module) => DefInstance(info, name, module)
    case connectRegex(args)      =>
      val Seq(loc, expr) = splitTop(args)
      Connect(info, parseExpr(loc), parseExpr(expr))
    case whenRegex(pred) => Conditionally(info, parseExpr(pred), toStatements(kids), Nil)
    case other           => OtherStmt(info, other, toStatements(kids))
  }
}

def toModule(node: Node): DefModule = {
  val Node(Line(_, text, info), kids) = node
  val moduleRegex(pub, name)          = text
  val (portNodes, bodyNodes)          = kids.partition(k => portRegex.matches(k.line.text))
  val ports                           = portNodes.map { k =>
    val portRegex(dir, pname, tpe) = k.line.text
    Port(k.line.info, dir, pname, tpe)
  }
  DefModule(info, name, pub != null, ports, toStatements(bodyNodes))
}

Step 4: the circuit. The first lines (the version and the annotations in `%[[ ... ]]`) are skipped, and the `layer` declarations are ignored. Modules are indented by two spaces, so that is the indentation of the first level of our tree.

In [ ]:
def parseCircuit(firrtl: String): Circuit = {
  // Skip the "FIRRTL version" line and the multi-line annotation block `circuit Top :%[[ ... ]]`.
  val lines         = firrtl.linesIterator.dropWhile(!_.startsWith("circuit ")).toList
  val header        = lines.head
  val afterAnnos    = if (header.endsWith("%[[")) lines.tail.dropWhile(_.trim != "]]").tail else lines.tail
  val main          = header.stripPrefix("circuit ").takeWhile(c => c != ' ' && c != ':')
  val (children, _) = toTree(toLines(afterAnnos.mkString("\n")), 2)
  // `layer` and `extmodule` declarations are not analyzed: we keep their text to serialize it back.
  def rawLines(n: Node, depth: Int): Seq[String] =
    (("  " * depth) + n.line.text + (if (n.line.info.isEmpty) "" else s" @[${n.line.info}]")) +: n.kids.flatMap(
      rawLines(_, depth + 1)
    )
  val (modules, others) = children.partition(n => moduleRegex.matches(n.line.text))
  Circuit(main, modules.map(toModule), others.flatMap(rawLines(_, 1)))
}

Let's try it on the `DelayBy2` module from the previous section:

In [ ]:
class DelayBy2(width: Int) extends Module {
  val io = IO(new Bundle {
    val in  = Input(UInt(width.W))
    val out = Output(UInt(width.W))
  })
  val r0 = RegNext(io.in)
  val r1 = RegNext(r0)
  io.out := r1
}

lazy val delayAST = parseCircuit(getFirrtl(new DelayBy2(4)))
println(showTree(delayAST))

This is the data structure that holds the FIRRTL AST. It is a tree structure whose root node is a **Circuit**, which has the name of the top module and a list of **DefModule**s. Our module has three **Port**s (`clock`, `reset` and `io`), two registers and three connections. As in the previous section, the `info` fields hold the source locator that tells the line of Scala that produced a node.

A parser is only useful if it does not lose information. We can check this by *serializing* the AST back to text , and parsing again: the result must be the same tree. The `serialize` functions below are the mirror image of the parser, one `match` per kind of node; there is nothing new in them.

In [ ]:
def serialize(e: Expression): String = e match {
  case Reference(name)          => name
  case Literal(tpe, value)      => s"$tpe($value)"
  case DoPrim(op, args, consts) => (args.map(serialize) ++ consts).mkString(s"$op(", ", ", ")")
}

def serialize(s: Statement, indent: Int): Seq[String] = {
  val pad = "  " * indent
  val loc = (i: String) => if (i.isEmpty) "" else s" @[$i]"
  s match {
    case DefRegister(i, n, t, c)            => Seq(s"${pad}reg $n : $t, ${serialize(c)}${loc(i)}")
    case DefRegisterReset(i, n, t, c, r, v) =>
      Seq(s"${pad}regreset $n : $t, ${serialize(c)}, ${serialize(r)}, ${serialize(v)}${loc(i)}")
    case DefWire(i, n, t)          => Seq(s"${pad}wire $n : $t${loc(i)}")
    case DefNode(i, n, v)          => Seq(s"${pad}node $n = ${serialize(v)}${loc(i)}")
    case DefInstance(i, n, m)      => Seq(s"${pad}inst $n of $m${loc(i)}")
    case Connect(i, l, r)          => Seq(s"${pad}connect ${serialize(l)}, ${serialize(r)}${loc(i)}")
    case Conditionally(i, p, c, a) =>
      Seq(s"${pad}when ${serialize(p)} :${loc(i)}") ++ c.flatMap(serialize(_, indent + 1)) ++
        (if (a.isEmpty) Nil else Seq(s"${pad}else :") ++ a.flatMap(serialize(_, indent + 1)))
    case OtherStmt(i, t, kids) => Seq(s"$pad$t${loc(i)}") ++ kids.flatMap(serialize(_, indent + 1))
  }
}

def serialize(m: DefModule): Seq[String] = {
  val loc = (i: String) => if (i.isEmpty) "" else s" @[$i]"
  Seq(s"  ${if (m.isPublic) "public " else ""}module ${m.name} :${loc(m.info)}") ++
    m.ports.map(p => s"    ${p.direction} ${p.name} : ${p.tpe}${loc(p.info)}") ++ Seq("") ++
    m.body.flatMap(serialize(_, 2)) ++ Seq("")
}

def serialize(c: Circuit): String =
  (Seq(s"circuit ${c.main} :") ++ c.otherDecls ++ c.modules.flatMap(serialize)).mkString("\n")

In [ ]:
println(serialize(delayAST))
assert(parseCircuit(serialize(delayAST)) == delayAST)

## Traversal
### Understanding the map function

Writing an analysis usually requires writing functions which walk the AST to either collect information or replace nodes with new nodes.

The IR datastructure is a tree, where each IR node can have some number of children nodes (which in turn can have more children nodes, etc.). IR nodes without children are called leaves.

To write a function that traverses a `Circuit`, we need to first understand the functional programming concept `map`.

#### Understanding Seq.map
A Scala sequence of strings, can be represented as a tree with a root node `Seq` and children nodes `"a"`, `"b"`, and `"c"`:
```scala
val s = Seq("a", "b", "c")
```
```
    Seq
 /   |   \
"a" "b" "c"
```

Suppose we define a function `f` that, given a String argument `x`, concatenates `x` with itself:
```scala
def f(x: String): String = x + x
```

We can call `s.map` to return a new `Seq[String]` whose children are the result of applying `f` to every child of s:
```scala
val s = Seq("a", "b", "c")
def f(x: String): String = x + x  // repeated declaration for clarity
val t = s.map(f)
println(t) // List("aa", "bb", "cc")
```
```
     Seq
 /    |    \
"aa" "bb" "cc"
```

#### Our map functions

We use this "mapping" idea to create our own, custom `map` methods on IR nodes. Suppose we have a `DoPrim` expression representing 1 + 1; this can be depicted as a tree of expressions with a root node `DoPrim`:
```
        DoPrim
     /          \
 Literal      Literal
```

If we have a function `f` that takes an `Expression` argument and returns a new `Expression`, we can "map" it onto all children `Expression` of a given IR node, like our `DoPrim`. This would return the following new `DoPrim`, whose children are the result of applying `f` to every `Expression` child of `DoPrim`:
```
        DoPrim
     /          \
 f(Literal)    f(Literal)
```

Sometimes IR nodes have children of multiple types. For example, `Conditionally` has both `Expression` and `Statement` children. This is why we have two methods, `mapExpr` and `mapStmt`:
```scala
val c = Conditionally(info, e, Seq(s1), Seq(s2)) // e: Expression, s1, s2: Statement, info: String
def fExp(e: Expression): Expression = ...
def fStmt(s: Statement): Statement = ...
c.mapExpr(fExp)  // Conditionally(info, fExp(e), Seq(s1), Seq(s2))
c.mapStmt(fStmt) // Conditionally(info, e, Seq(fStmt(s1)), Seq(fStmt(s2)))
```
You can look at their implementation in the case class definitions above. Note that they apply the function only to the *direct* children, they do not recurse. Recursion is up to the function we write, as we will see now.

Scala has "infix notation", which allows you to drop the `.` and parentheses when calling a function which has one argument. Often, we write such calls with infix notation: `c mapExpr fExp` is equivalent to `c.mapExpr(fExp)`.

### Pre-order traversal

To traverse the tree, we use `map` to write recursive functions which visit every child of every node we care about.

Suppose we want to collect the names of every register declared in the design; we know this requires visiting every `Statement`. However, some `Statement` nodes can have children `Statement`s (the body of a `when`). Thus, we need to write a function that will both check if its input argument is a register and, if not, will recursively apply itself to all `Statement` children of its input argument.

The following function `collectRegs` takes two arguments: a mutable set of register names, and a `Statement`. Using function currying, we can pass only the first argument to return a new function with the desired type signature (`Statement => Statement`):

In [ ]:
import scala.collection.mutable

def collectRegs(regNames: mutable.LinkedHashSet[String])(s: Statement): Statement = s match {
  // If register, add name to regNames
  case r: IsRegister =>
    regNames += r.name
    r // Return argument unchanged (ok because registers have no Statement children)
  // If not, apply collectRegs(regNames) to all children Statement
  case _ => s.mapStmt(collectRegs(regNames)) // Note that collectRegs(regNames) is of type Statement => Statement
}

// A module with a register inside a `when`, to make it interesting
class WhenReg extends Module {
  val io = IO(new Bundle { val en = Input(Bool()); val d = Input(UInt(4.W)); val q = Output(UInt(4.W)) })
  val a  = Reg(UInt(4.W))
  io.q := a
  when(io.en) {
    val b = RegNext(io.d)
    a := b + 1.U
  }
}

lazy val whenRegAST = parseCircuit(getFirrtl(new WhenReg))
val regNames        = mutable.LinkedHashSet[String]()
whenRegAST.modules.foreach(m => m.mapStmt(collectRegs(regNames)))
println(regNames)

This pattern is very common, and is called "pre-order traversal" because the recursive function matches on the original IR node before recursively applying to its children nodes. Note that we call `m.mapStmt` only for its side effect on `regNames`, and ignore the new module it returns.

### Post-order traversal

We can write the previous example in a "post-order traversal" as follows:

In [ ]:
def collectRegsPost(regNames: mutable.LinkedHashSet[String])(s: Statement): Statement =
  // Note we immediately recurse to the children nodes, then match
  s.mapStmt(collectRegsPost(regNames)) match {
    // If register, add name to regNames
    case r: IsRegister =>
      regNames += r.name
      r
    // If not, return s
    case other => other // Note that all Statement children of s have had collectRegsPost(regNames) already applied
  }

val regNames2 = mutable.LinkedHashSet[String]()
whenRegAST.modules.foreach(m => m.mapStmt(collectRegsPost(regNames2)))
println(regNames2)

While the traversal ordering is different between these two examples, it makes no difference for this use case (and many others). However, it is an important tool to keep in your back pocket for when the traversal ordering matters.

### The collection-style traversal
Mutable sets and `map` calls that throw away their result are a bit clumsy for pure analysis. Since our tree is an ordinary Scala data structure, we can also *flatten* it into a sequence and use the whole collections library. Let's define three helpers:

* `allStatements(body)`: every statement of a body, including those nested in `when`s;
* `directExprs(stmt)`: the expressions that appear directly in a statement;
* `allExprs(expr)`: an expression and all of its sub-expressions.

In [ ]:
// All statements of a body, including the ones nested inside `when`s (pre-order).
def allStatements(body: Seq[Statement]): Seq[Statement] = body.flatMap {
  case w: Conditionally => w +: (allStatements(w.conseq) ++ allStatements(w.alt))
  case o: OtherStmt     => o +: allStatements(o.children)
  case s                => Seq(s)
}

// The Expressions that appear directly in a statement (not the sub-expressions).
def directExprs(s: Statement): Seq[Expression] = s match {
  case r: DefRegister      => Seq(r.clock)
  case r: DefRegisterReset => Seq(r.clock, r.reset, r.init)
  case n: DefNode          => Seq(n.value)
  case c: Connect          => Seq(c.loc, c.expr)
  case w: Conditionally    => Seq(w.pred)
  case _                   => Nil
}

// An Expression and all its sub-expressions (pre-order).
def allExprs(e: Expression): Seq[Expression] = e match {
  case DoPrim(_, args, _) => e +: args.flatMap(allExprs)
  case leaf               => Seq(e)
}

With them, "collect all registers" becomes a one-liner:

In [ ]:
for (m <- whenRegAST.modules) {
  println(m.name + ": " + allStatements(m.body).collect { case r: IsRegister => r.name })
}

...and "which primitive operations does this module use, and how often?" is not much longer. We take all statements, all of their expressions, all of the sub-expressions, and keep the `DoPrim`s:

In [ ]:
for (m <- whenRegAST.modules) {
  val ops = allStatements(m.body).flatMap(directExprs).flatMap(allExprs).collect { case DoPrim(op, _, _) => op }
  println(m.name + ": " + ops.groupBy(identity).map { case (op, l) => op -> l.size })
}

<span style="color:red">**Exercise: Operators used**</span><br>
Write a function `opsUsed` that returns the set of names of the primitive operations that a module uses (e.g. `Set("add", "tail", ...)`). It should find the operations inside of `when` blocks too. We test it with a module that computes `(a + b) ^ a` (the `+` is a `tail(add(...), 1)` in FIRRTL).

In [ ]:
def opsUsed(m: DefModule): Set[String] = ???

class OpsDemo extends Module {
  val io = IO(new Bundle { val a = Input(UInt(4.W)); val b = Input(UInt(4.W)); val out = Output(UInt(4.W)) })
  io.out := (io.a + io.b) ^ io.a
}
lazy val opsDemoAST = parseCircuit(getFirrtl(new OpsDemo))
println(opsUsed(opsDemoAST.modules.head))
assert(opsUsed(opsDemoAST.modules.head) == Set("add", "tail", "xor"))
println("SUCCESS!!")

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-1" />
<label for="check-1"><strong>Solution</strong> (click to toggle displaying)</label>
<article>
<pre style="background-color:#f7f7f7">
def opsUsed(m: DefModule): Set[String] =
  allStatements(m.body).flatMap(directExprs).flatMap(allExprs).collect { case DoPrim(op, _, _) => op }.toSet
</pre></article></div></section></div>

That's it for this section! In the next section, we will see how analyses and transformations of a design are commonly structured.

# You're done!

[Return to the top.](#top)